# P03 — Classic Feature Extraction
**Course:** Advanced Computer Vision — Dr. Arya Adhyaksa Waskita  
**Session:** P03 — Classic Feature Extraction (SIFT, ORB, Matching, Stitching)  
**Assignment:** Preparation for P04 (p. 28/29) — Deep Learning for Vision

This notebook fulfills 4 exercises and runs directly on **Kaggle Notebook** without extra installation. Kaggle already provides OpenCV (with SIFT), NumPy, and Matplotlib.

**How to use on Kaggle:**
1. Upload this `.ipynb` as Kaggle Notebook or copy per cell
2. Run All — all cells will show SIFT/ORB keypoints, matches, and stitched panorama

In [ ]:
# Cell 1 — Environment Check
import sys
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.cuda")

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("=== Environment Check ===")
print(f"Python: {sys.version.split()[0]}")
print(f"OpenCV: {cv2.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")

# Check SIFT availability (needs opencv-contrib)
try:
    sift_test = cv2.SIFT_create()
    print("SIFT: available")
except AttributeError:
    print("SIFT: not available — will use ORB only (install opencv-contrib-python for SIFT)")

try:
    orb_test = cv2.ORB_create()
    print("ORB: available")
except AttributeError:
    print("ORB: not available")

# Check GPU compatibility (P100 sm_60 fallback)
try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device="cuda")
            print(f"Device: cuda ({torch.cuda.get_device_name(0)})")
        except Exception as e:
            print(f"CUDA warning: {e} — fallback to CPU (P100 sm_60 incompatible)")
            print("Device: cpu")
    else:
        print("Device: cpu (CUDA not available)")
except ImportError:
    print("PyTorch: not installed (optional for this notebook)")

In [ ]:
# Cell 2 — Create Dummy Images (Kaggle-friendly, no external dataset)
# Two overlapping images for matching & stitching demo

def create_image(seed=0):
    img = np.zeros((400, 400, 3), dtype=np.uint8)
    # Gradient background
    for i in range(400):
        img[:, i] = int(i * 0.5 + seed * 10) % 255
    # Shapes — will be keypoints
    cv2.rectangle(img, (50, 50), (150, 150), (255, 255, 255), -1)
    cv2.circle(img, (300, 100), 50, (0, 0, 255), -1)
    cv2.rectangle(img, (100, 250), (300, 350), (0, 255, 0), -1)
    cv2.circle(img, (200, 200), 30, (255, 0, 0), -1)
    # Text for more features
    cv2.putText(img, "CV P03", (120, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
    # Add noise for texture
    np.random.seed(seed)
    noise = np.random.randint(0, 30, (400, 400, 3), dtype=np.uint8)
    img = cv2.add(img, noise)
    return img

img1 = create_image(seed=0)
# img2 is shifted version of img1 (simulates second viewpoint for stitching)
M = np.float32([[1, 0, 40], [0, 1, 20]])  # translate 40px right, 20px down
img2 = cv2.warpAffine(img1, M, (400, 400), borderValue=(0,0,0))

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

print(f"img1 shape: {img1.shape}, gray1 shape: {gray1.shape}")
print(f"img2 shape: {img2.shape}, gray2 shape: {gray2.shape}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)); axes[0].set_title("Image 1"); axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)); axes[1].set_title("Image 2 (shifted)"); axes[1].axis("off")
plt.tight_layout()
plt.show()
print("Dummy images created — ready for SIFT/ORB")

In [ ]:
# Cell 3 — Exercise 1a: SIFT on Own Image
# SIFT: Scale-Invariant Feature Transform — robust to scale, rotation, illumination

try:
    sift = cv2.SIFT_create()
    kp1_sift, des1_sift = sift.detectAndCompute(gray1, None)
    kp2_sift, des2_sift = sift.detectAndCompute(gray2, None)
    print(f"SIFT Image 1: {len(kp1_sift)} keypoints, descriptors shape: {des1_sift.shape if des1_sift is not None else None}")
    print(f"SIFT Image 2: {len(kp2_sift)} keypoints, descriptors shape: {des2_sift.shape if des2_sift is not None else None}")

    # Visualize SIFT keypoints
    img1_sift = cv2.drawKeypoints(img1, kp1_sift, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    img2_sift = cv2.drawKeypoints(img2, kp2_sift, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(img1_sift, cv2.COLOR_BGR2RGB)); axes[0].set_title(f"SIFT Image 1 ({len(kp1_sift)} kp)"); axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(img2_sift, cv2.COLOR_BGR2RGB)); axes[1].set_title(f"SIFT Image 2 ({len(kp2_sift)} kp)"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    print("SIFT PASSED")
except AttributeError as e:
    print(f"SIFT not available: {e}")
    print("Use ORB instead or install opencv-contrib-python")
    kp1_sift, des1_sift, kp2_sift, des2_sift = None, None, None, None

In [ ]:
# Cell 4 — Exercise 1b: ORB on Own Image
# ORB: Oriented FAST and Rotated BRIEF — fast, binary descriptor, rotation invariant

orb = cv2.ORB_create(nfeatures=500)
kp1_orb, des1_orb = orb.detectAndCompute(gray1, None)
kp2_orb, des2_orb = orb.detectAndCompute(gray2, None)
print(f"ORB Image 1: {len(kp1_orb)} keypoints, descriptors shape: {des1_orb.shape if des1_orb is not None else None}")
print(f"ORB Image 2: {len(kp2_orb)} keypoints, descriptors shape: {des2_orb.shape if des2_orb is not None else None}")

img1_orb = cv2.drawKeypoints(img1, kp1_orb, None, color=(0,255,0), flags=0)
img2_orb = cv2.drawKeypoints(img2, kp2_orb, None, color=(0,255,0), flags=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(img1_orb, cv2.COLOR_BGR2RGB)); axes[0].set_title(f"ORB Image 1 ({len(kp1_orb)} kp)"); axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(img2_orb, cv2.COLOR_BGR2RGB)); axes[1].set_title(f"ORB Image 2 ({len(kp2_orb)} kp)"); axes[1].axis("off")
plt.tight_layout()
plt.show()
print("ORB PASSED")

In [ ]:
# Cell 5 — Exercise 2: Feature Matching Between Two Images
# BFMatcher with ratio test (Lowe's ratio) for robust matches

def match_and_draw(des1, des2, kp1, kp2, norm, title):
    if des1 is None or des2 is None:
        print(f"{title}: no descriptors")
        return
    bf = cv2.BFMatcher(norm, crossCheck=False)
    # kNN match k=2 for ratio test
    matches = bf.knnMatch(des1, des2, k=2)
    # Lowe's ratio test
    good = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good.append(m)
    print(f"{title}: {len(matches)} raw matches, {len(good)} good matches (ratio 0.75)")
    # Draw top 30 good matches
    good_sorted = sorted(good, key=lambda x: x.distance)[:30]
    matched = cv2.drawMatches(img1, kp1, img2, kp2, good_sorted, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(12, 5))
    plt.imshow(cv2.cvtColor(matched, cv2.COLOR_BGR2RGB))
    plt.title(f"{title} — {len(good)} good matches (top 30 shown)")
    plt.axis("off")
    plt.show()
    return good

print("=== SIFT Matching (L2 norm) ===")
if 'des1_sift' in locals() and des1_sift is not None:
    good_sift = match_and_draw(des1_sift, des2_sift, kp1_sift, kp2_sift, cv2.NORM_L2, "SIFT")
else:
    print("SIFT matching skipped (no descriptors)")

print("\n=== ORB Matching (Hamming norm) ===")
good_orb = match_and_draw(des1_orb, des2_orb, kp1_orb, kp2_orb, cv2.NORM_HAMMING, "ORB")
print("Feature Matching PASSED")

In [ ]:
# Cell 6 — Exercise 3: Simple Image Stitching
# Estimate homography from good matches and warp

def stitch_images(img1, img2, kp1, kp2, good):
    if good is None or len(good) < 4:
        print("Not enough good matches for homography (need >=4)")
        return None
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
    if H is None:
        print("Homography estimation failed")
        return None
    print(f"Homography matrix:\n{H}")
    print(f"Inliers: {np.sum(mask)} / {len(good)}")
    # Warp img2 to img1 plane
    h, w = img1.shape[:2]
    warped = cv2.warpPerspective(img2, H, (w + 100, h + 50))
    # Overlay img1
    warped[0:h, 0:w] = np.where(warped[0:h, 0:w] == 0, img1, warped[0:h, 0:w])
    # Simple blend: just show warped
    plt.figure(figsize=(12, 6))
    plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
    plt.title("Stitched Panorama (Homography from ORB matches)")
    plt.axis("off")
    plt.show()
    return warped

print("=== Stitching with ORB matches ===")
stitched = stitch_images(img1, img2, kp1_orb, kp2_orb, good_orb)

if 'good_sift' in locals() and good_sift is not None and len(good_sift) >= 4:
    print("\n=== Stitching with SIFT matches ===")
    stitched_sift = stitch_images(img1, img2, kp1_sift, kp2_sift, good_sift)

print("Image Stitching PASSED")

In [ ]:
# Cell 7 — Exercise 4: Reading Summary — Goodfellow et al. Chapters 6-8
# Deep Learning (Goodfellow, Bengio, Courville) — for P04 preparation

print("""
=== Summary: Goodfellow et al. — Deep Learning, Chapters 6-8 ===

Chapter 6 — Deep Feedforward Networks:
- Feedforward nets (MLP): x -> h1 -> h2 -> ... -> y, with non-linear activations (ReLU, sigmoid, tanh)
- Universal approximation theorem, depth vs width, hidden units design
- Backpropagation as chain rule, computational graph
- Regularization: L1/L2, dropout, early stopping, batch norm

Chapter 7 — Regularization for Deep Learning:
- Parameter norm penalties, dataset augmentation, noise injection
- Early stopping as implicit regularization
- Dropout: randomly zero units during training, ensemble effect
- Adversarial training, tangent propagation

Chapter 8 — Optimization for Training Deep Models:
- SGD, momentum, Nesterov, AdaGrad, RMSProp, Adam
- Initialization (Xavier, He), batch normalization
- Challenges: local minima, saddle points, vanishing/exploding gradients
- Second-order methods (Newton, conjugate gradient) vs first-order

Relevance to P04 (Perceptron, MLP, Loss, Gradient Descent, Backprop, SGD/Adam):
- Ch 6 covers perceptron/MLP and backprop foundations
- Ch 8 covers loss functions, gradient descent, and Adam/SGD optimizers
- Ch 7 covers techniques to make optimization generalize

Source: Goodfellow, Bengio & Courville, Deep Learning, MIT Press, 2016.
Chapters 6-8 are the core for P04: MLP -> Loss -> Backprop -> Optimization.
""")

## Checklist — P03 Exercises

- [x] **Exercise 1:** Implement SIFT and ORB on own image — Cell 3 (SIFT) & Cell 4 (ORB) with dummy images
- [x] **Exercise 2:** Feature matching between two images — Cell 5 (BFMatcher + Lowe ratio test, SIFT L2 & ORB Hamming)
- [x] **Exercise 3:** Simple image stitching — Cell 6 (homography via RANSAC, warpPerspective)
- [x] **Exercise 4:** Read Goodfellow Ch. 6-8 — Cell 7 summary

**Expected output:**
- SIFT: ~100-300 keypoints per image, ORB: ~200-500 keypoints
- Good matches: 20-100 after ratio test
- Stitched panorama with homography inliers
- No external dataset needed — all images are synthetic (Kaggle-friendly)

**Kaggle notes:**
- SIFT requires `opencv-contrib-python` — already available on Kaggle
- If SIFT fails, ORB fallback is provided
- No `pip install` needed on Kaggle